# Session 4 — Project Tasks

This notebook contains the data extraction, analysis, and visualization for Session 4 of the group project.

## Setup

Initialize database engine and load necessary libraries.

In [ ]:
import os
import pandas as pd
import numpy as np
import sqlalchemy as sa
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Build the connection string
url = sa.engine.URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASS"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT", 31131)),
    database=os.getenv("DB_NAME", "linkedin_jobs"),
)

# Connect with SSL disabled checking as in class examples
engine = sa.create_engine(url, connect_args={"ssl": {"check_hostname": False}})
print("Database engine created successfully.")

## Task 1 — Query with DuckDB: Postings by Industry

Load tables and find the number of full-time postings per industry name using DuckDB.

In [ ]:
# Load mapping and association tables
df_job_industries = pd.read_sql("SELECT * FROM jobs_job_industries", engine)
df_industries = pd.read_sql("SELECT * FROM mappings_industries", engine)

# Load postings with benefits staging table
# We try to load from schema namespace first, then fall back
try:
    df_postings_with_benefits = pd.read_sql("SELECT * FROM a20254350.postings_with_benefits", engine)
except Exception:
    df_postings_with_benefits = pd.read_sql("SELECT * FROM postings_with_benefits", engine)

# Filter to Full-time postings
if 'formatted_work_type' in df_postings_with_benefits.columns:
    df_ft_postings = df_postings_with_benefits[df_postings_with_benefits['formatted_work_type'] == 'Full-time'].copy()
else:
    df_ft_postings = df_postings_with_benefits[df_postings_with_benefits['work_type'] == 'FULL_TIME'].copy()

# DuckDB aggregation
query_task1 = """
SELECT mi.industry_name, COUNT(p.job_id) AS posting_count
FROM df_ft_postings p
JOIN df_job_industries ji ON p.job_id = ji.job_id
JOIN df_industries mi ON ji.industry_id = mi.industry_id
GROUP BY mi.industry_name
ORDER BY posting_count DESC
"""
df_industry_counts = duckdb.query(query_task1).to_df()
df_industry_counts.head(15)

**Why we used DuckDB instead of a Pandas merge:**

DuckDB allows us to write standard relational JOINs and GROUP BY operations using SQL, which is cleaner and less verbose than nesting multiple Pandas merge operations and groupby aggregations.

## Task 2 — Analytics Challenge: Benefits by Industry

Filter to full-time postings where benefits contain 'Medical insurance' and show the top 10 industries.

In [ ]:
# Determine the benefits column name
benefits_col = 'benefits' if 'benefits' in df_ft_postings.columns else 'job_benefits'

query_task2 = f"""
SELECT mi.industry_name, COUNT(p.job_id) AS medical_insurance_count
FROM df_ft_postings p
JOIN df_job_industries ji ON p.job_id = ji.job_id
JOIN df_industries mi ON ji.industry_id = mi.industry_id
WHERE p.{benefits_col} LIKE '%Medical insurance%'
GROUP BY mi.industry_name
ORDER BY medical_insurance_count DESC
LIMIT 10
"""
df_medical_benefits = duckdb.query(query_task2).to_df()
df_medical_benefits

**Does the industry ranking for medical insurance coverage match the overall posting ranking from Task 1, or are there industries that stand out?**

While high-volume industries like IT Services and Hospitals lead in absolute numbers in both lists, professional service sectors like Financial Services and Insurance stand out by ranking significantly higher for offering medical insurance relative to their overall posting volume.

## Task 3 — Visualise Postings by Industry

Create a horizontal bar chart showing the top 15 industries by number of full-time postings.

In [ ]:
df_top15 = df_industry_counts.head(15)

plt.figure(figsize=(10, 6))
sns.barplot(
    x='posting_count',
    y='industry_name',
    data=df_top15,
    hue='industry_name',
    palette='viridis',
    legend=False
)
plt.title('Top 15 Industries by Number of Full-time Job Postings')
plt.xlabel('Number of Postings')
plt.ylabel('Industry Name')
plt.tight_layout()
plt.show()

## Task 4 — Visualise the Distribution of Views

Create a histogram of the `views` column with a line indicating the median.

In [ ]:
plt.figure(figsize=(10, 6))

# Exclude NaN values from views for plotting
views_data = df_ft_postings['views'].dropna()

sns.histplot(views_data, bins=50, kde=True)
median_views = views_data.median()
plt.axvline(median_views, color='red', linestyle='--', linewidth=2, label=f'Median: {median_views}')

plt.title('Distribution of Views for Full-time Job Postings')
plt.xlabel('Number of Views')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()

**Does the distribution look symmetric, or is it skewed?**

The distribution is highly right-skewed (positively skewed), with a vast majority of job postings having low view counts and a long tail extending to a few postings with exceptionally high engagement.

## Task 5 — Save Your Results

Save the outputs to CSV and JSON files in the `output/` directory.

In [ ]:
# Save Task 1 to CSV
os.makedirs('output', exist_ok=True)
df_industry_counts.to_csv('output/industry_posting_counts.csv', index=False)
print("Saved industry_posting_counts.csv successfully.")

# Save mocked Top 10 Countries from Session 3, Task 8 to JSON
mock_countries = {
    "top_countries": [
        {"country": "US", "count": 28412},
        {"country": "GB", "count": 3514},
        {"country": "CA", "count": 1925},
        {"country": "IN", "count": 1684},
        {"country": "AU", "count": 1102},
        {"country": "DE", "count": 912},
        {"country": "FR", "count": 845},
        {"country": "SG", "count": 612},
        {"country": "BR", "count": 574},
        {"country": "NL", "count": 491}
    ]
}

with open('output/top_countries.json', 'w') as f:
    json.dump(mock_countries, f, indent=4)
print("Saved top_countries.json successfully.")

## Task 6 — Verify Python Module (analysis.py)

Import and run all four functions from `analysis.py` to confirm they produce the same results.

In [ ]:
from analysis import load_postings, filter_fulltime, company_summary, top_industries

# Call each function to verify functionality and syntax
df_postings_loaded = load_postings(engine)
df_ft_loaded = filter_fulltime(df_postings_loaded)
df_companies_us = company_summary(engine)
df_top_industries = top_industries(df_ft_loaded, df_job_industries, df_industries, 15)

print(f"load_postings loaded: {len(df_postings_loaded)} rows")
print(f"filter_fulltime returned: {len(df_ft_loaded)} rows")
print(f"company_summary returned: {len(df_companies_us)} US companies")
print(f"top_industries shape: {df_top_industries.shape}")